In [1]:
%pip install --upgrade pip
%pip install torch --index-url https://download.pytorch.org/whl/cpu
%pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
MODEL_DIR = Path("smollm2_model")

print("Downloading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Downloading model...")

model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

print("Saving model locally...")

tokenizer.save_pretrained(MODEL_DIR)
model.save_pretrained(MODEL_DIR, safe_serialization=True)

print("Model downloaded successfully!")
print("Model location:", MODEL_DIR.resolve())

C:\Users\Akash\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Akash\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  724MB            

model.safetensors: downloading bytes:           |  0.00B            

C:\Users\Akash\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Akash\.cache\huggingface\hub\models--HuggingFaceTB--SmolLM2-360M-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Saving model locally...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model downloaded successfully!
Model location: C:\Users\Akash\python\smollm2_model


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.set_num_threads(4)

MODEL_DIR = "smollm2_model"

print("Loading local model...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True
)

model.to("cpu")
model.eval()

print("Model loaded successfully!")
print("Running on CPU.")

Loading local model...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!
Running on CPU.


In [4]:
SYSTEM_PROMPT = """
You are a helpful AI teaching assistant.
Explain concepts in simple English.
Use examples whenever helpful.
If you are unsure about an answer, say so.
"""

MAX_NEW_TOKENS = 120
MAX_HISTORY_MESSAGES = 6


def generate_response(user_message, history=None):

    if history is None:
        history = []

    # Keep recent conversation messages
    recent_history = history[-MAX_HISTORY_MESSAGES:]

    # Prepare the conversation
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ] + recent_history + [
        {
            "role": "user",
            "content": user_message
        }
    ]

    # Convert conversation into model input
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    # Generate a response
    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Extract only newly generated tokens
    input_length = inputs["input_ids"].shape[1]

    new_tokens = output_ids[0][input_length:]

    # Convert tokens to readable text
    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    if not answer:
        answer = "I could not generate a response. Please try again."

    return answer


print("Chatbot function created successfully!")

Chatbot function created successfully!


In [5]:
question = "Explain Generative AI in simple terms."

answer = generate_response(question)

print("You:", question)
print("\nGenAI Chatbot:", answer)

You: Explain Generative AI in simple terms.

GenAI Chatbot: Generative AI is like having a magic wand that can create anything it's told to make. It's called generative because it's able to generate new things or images without being programmed to do so. Think of it as drawing pictures or creating music on your own. 

Imagine if you had a favorite hobby, like painting or playing the guitar. Now imagine that you could just sit down and start creating something entirely new, without needing any instructions at all. That's what generative AI does - it allows humans to create new content or ideas without knowing how they will look or sound.


In [6]:
chat_history = []

question = "What is Machine Learning?"

answer = generate_response(question, chat_history)

print("You:", question)
print("\nBot:", answer)

chat_history.extend([
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
])

You: What is Machine Learning?

Bot: Machine learning is like teaching computers to learn and make decisions just like humans do. Imagine if you could teach a computer how to recognize pictures of cats or dogs using lots of examples from other photos. That's basically what machine learning does, but with data instead of images!

Machine learning helps computers become smarter by analyzing large amounts of information and figuring out patterns. This allows them to perform tasks that were previously only done by people, such as predicting which weather will be sunny next week, recognizing faces in photos, or even understanding human language.

Think of it like this: when we ask


In [7]:
question = "Give me a real-life example of it."

answer = generate_response(question, chat_history)

print("You:", question)
print("\nBot:", answer)

chat_history.extend([
    {"role": "user", "content": question},
    {"role": "assistant", "content": answer}
])

You: Give me a real-life example of it.

Bot: A great example of machine learning at work is self-driving cars. These cars use sensors on their roofs to detect the world around them and then use algorithms (special instructions) to figure out how to navigate safely. The car might look for roads, traffic lights, pedestrians, and other obstacles, all without needing any training from a human driver. This makes the car much safer than regular cars because they can react quickly to changing situations.


In [8]:
def start_chat():

    history = []

    print("=" * 50)
    print("       OFFLINE GENAI CHATBOT")
    print("=" * 50)
    print("Ask any question.")
    print("Type /reset to clear history.")
    print("Type /exit to stop.")
    print("=" * 50)

    while True:

        question = input("\nYou: ").strip()

        if not question:
            continue

        if question.lower() == "/exit":
            print("Bot: Goodbye!")
            break

        if question.lower() == "/reset":
            history.clear()
            print("Bot: Conversation history cleared.")
            continue

        print("Bot: Thinking...")

        answer = generate_response(question, history)

        print("\nBot:", answer)

        history.extend([
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ])

        # Keep the history bounded
        if len(history) > 12:
            history = history[-12:]


# Run the chatbot
start_chat()

       OFFLINE GENAI CHATBOT
Ask any question.
Type /reset to clear history.
Type /exit to stop.



You:  what is the sql


Bot: Thinking...

Bot: SQL (Structured Query Language) is a programming language used to manage and access databases. It helps users control how data is stored, retrieved, updated, and deleted from databases. SQL statements can be written to insert, update, or delete data within a database. 

For example, let's consider the following SQL query:

```sql
SELECT * FROM customers WHERE name = 'John Doe';
```

This query asks the database to display all records of "customers" who have a name equal to "John Doe". The result will show all columns (`*



You:  what is python


Bot: Thinking...

Bot: Python is a high-level programming language that was created by Guido van Rossum in 1989. It is known for its simplicity, readability, and versatility. Python is often used as a first language for beginners due to its syntax being similar to other languages like JavaScript and HTML/CSS. 

One of the key features of Python is its extensive libraries and frameworks, such as NumPy, pandas, and scikit-learn, which make it easy to perform various tasks like data analysis, machine learning, web development, and more.

Some common applications of Python include


...

Bot: Thinking...

Bot: Python is a popular and versatile programming language with many uses across various fields. Here are some ways to exit your session:

1. **Ctrl+D (Windows/Linux):**: Cmd+Z (MacOS) or Ctrl+X on Windows, or ⌥ + X on MacOS. This command terminates the current program.

2. **Terminal or Command Prompt**: To exit a terminal or command prompt, simply type `exit` and press Enter.

3. **Package Manager**: On Unix-based systems (like macOS or Linux), you can use `



You:  what is powerbi


Bot: Thinking...

Bot: Power BI (Business Intelligence Express) is a business intelligence (BI) tool designed specifically for data visualization and analytics. It offers a range of capabilities for creating interactive dashboards, reports, and stories using data from various sources, including relational databases, Excel files, and external APIs.

The primary goal of Power BI is to provide a user-friendly interface for exploring and analyzing large datasets to drive informed decision-making. By leveraging advanced analytical tools and creative design techniques, Power BI enables businesses to unlock insights and turn data into meaningful actions.

Key features of Power BI include:

1



You:  /exit


Bot: Goodbye!
